# 🎄 Shein Christmas reel — free Colab T4, open-source only

**تشغيل:** `Runtime → Change runtime type → T4 GPU`، وبعدها `Runtime → Run all`. النتيجة في الآخر: `/content/out/xmas-reel-colab.mp4`.

This notebook rebuilds Zizo's ~16 s vertical Shein Christmas reel end-to-end with **free, open models only** (no API keys, no paid services):

| Stage | Model | Why |
|---|---|---|
| 1. Voice | **VoiceTut-TTS** (OmniVoice fine-tuned on Egyptian Arabic) cloning the Gemini *Aoede* reference clip · fallback **Chatterbox Multilingual** (`ar`) · last resort: the original Aoede lines | Only open TTS we found that speaks Egyptian colloquial; 2.9 GB VRAM on T4. Two takes per line, best one picked by Whisper-large-v3-turbo CER |
| 2. Video | **LTX-Video 0.9.8 2B distilled** image-to-video (8 steps), 480×832, starting from the real avatar stills | Smallest maintained I2V that fits a T4 + 12 GB RAM; clothes come from the photo, not generated. T5 prompt embeddings are pre-computed (no 19 GB text encoder download). Fallback: Ken Burns |
| 3. Lip-sync | **LatentSync 1.5** (ByteDance, 256 px, fp16, DeepCache) | Diffusion lip-sync with good temporal stability; 8 GB VRAM → fits T4 (1.6 needs 18 GB). Fallback: un-synced clip |
| 4. Edit | ffmpeg: 1080×1920 / 30 fps, pre-rendered Arabic captions, our own Jingle-Bells bed (public-domain tune), loudnorm −14 LUFS | Same layout as the paid version |

Expected total on a free T4: **~30–45 min** (most of it is installs and ~20 GB of model downloads). Each stage prints its time and writes to `/content/out`.
Inputs are fetched from the public repo folder `ahmed88bite-maker/zizo-games/colab-reel/`.

In [ ]:
#@title 1) Setup — GPU check, config, inputs { display-mode: "form" }
import os, json, time, subprocess, base64, urllib.request, shutil, sys, glob
T_START = time.time(); TIMES = {}
WORK, OUT = "/content/work", "/content/out"
for d in (WORK + "/inputs", WORK + "/stages", OUT): os.makedirs(d, exist_ok=True)
os.environ.update(WORK=WORK, OUT=OUT, HF_HOME="/content/hf", TOKENIZERS_PARALLELISM="false",
                  PYTHONUNBUFFERED="1", HF_HUB_DISABLE_PROGRESS_BARS="1")
# Forced into EVERY stage subprocess: Colab exports MPLBACKEND=module://matplotlib_inline... which the stage venvs
# don't have (run 1 LatentSync crash); expandable segments reduce CUDA fragmentation on the 15 GB T4.
STAGE_ENV = {"MPLBACKEND": "Agg", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}

CONFIG = {
    "seed": 42,
    "tts_takes": 2,               # takes per line; best picked by Whisper CER
    "tts_steps": 32,
    "tts_pick_with_whisper": True,
    "width": 480, "height": 832,  # LTX resolution (multiples of 32, ~9:16)
    "fps": 25, "max_frames": 97,   # LatentSync ping-pong-loops a clip if the line is longer
    "ltx_dtype": "fp16",          # T4 has no fast bf16; auto-retries in bf16 if fp16 gives NaN/black frames
    "lipsync": True,
    "ls_steps": 20, "ls_guidance": 1.5,
    "captions": True, "music_gain": 0.22, "end_hold": 1.2,
    "vo_lead": 0.12, "vo_tail": 0.22,
}
json.dump(CONFIG, open(f"{WORK}/config.json", "w"), indent=1)

def sh(cmd, cwd=None, env=None, check=True):
    """Run a shell command and stream its output."""
    print("$", cmd if len(cmd) < 300 else cmd[:300] + " ...", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, **STAGE_ENV, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "it/s]" in line or "s/it]" in line:          # keep tqdm noise short
            if not line.rstrip().endswith("100%"): continue
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd[:200]}")
    return p.returncode

def done(stage, t):
    TIMES[stage] = round(time.time() - t, 1)
    print(f"\n⏱  {stage}: {TIMES[stage]/60:.1f} min  (total {(time.time()-T_START)/60:.1f} min)", flush=True)

t = time.time()
g = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True)
if g.returncode != 0 or not g.stdout.strip():
    raise SystemExit("❌ No GPU. Runtime → Change runtime type → T4 GPU, then Run all again.")
print("GPU:", g.stdout.strip())
print(subprocess.run("free -g | head -2; df -h /content | tail -1; python3 --version; ffmpeg -version | head -1",
                     shell=True, capture_output=True, text=True).stdout)

REPO = "https://raw.githubusercontent.com/ahmed88bite-maker/zizo-games/main/colab-reel"
B64_ASSETS = ["s1.jpg", "s2.jpg", "s3.jpg", "s4.jpg", "ref_aoede_6s.wav", "music.mp3", "ltx_prompt_embeds.npz",
              "cap1.png", "cap2.png", "cap3.png", "cap4.png",
              "aoede_line1.wav", "aoede_line2.wav", "aoede_line3.wav", "aoede_line4.wav"]
TEXT_ASSETS = ["script.json", "scenes.json"]
STAGES = ["tts_stage.py", "tts_chatterbox.py", "post_audio.py", "i2v_stage.py", "ls_patch.sh", "ls_runner.py", "assemble.py"]
def fetch(url, tries=4):
    for k in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=60) as r: return r.read()
        except Exception as e:
            if k == tries - 1: raise
            print("retry", url, e); time.sleep(3 * (k + 1))
for n in B64_ASSETS:
    open(f"{WORK}/inputs/{n}", "wb").write(base64.b64decode(fetch(f"{REPO}/assets/{n}.b64")))
for n in TEXT_ASSETS:
    open(f"{WORK}/inputs/{n}", "wb").write(fetch(f"{REPO}/assets/{n}"))
for n in STAGES:
    open(f"{WORK}/stages/{n}", "wb").write(fetch(f"{REPO}/stages/{n}"))
for n in sorted(os.listdir(f"{WORK}/inputs")):
    print(f"  {n:28s} {os.path.getsize(f'{WORK}/inputs/{n}')/1e3:8.0f} kB")
script = json.load(open(f"{WORK}/inputs/script.json"))
print("\nScript:"); [print(" ", i + 1, l) for i, l in enumerate(script["lines"])]
if shutil.which("uv") is None: sh("pip -q install uv")
done("1_setup", t)

In [ ]:
#@title 2) Stage 1 — Egyptian voice clone (VoiceTut-TTS → Chatterbox → original Aoede) { display-mode: "form" }
from IPython.display import Audio, display
t = time.time()
GEN = "/content/venvs/gen"   # one venv for TTS + LTX: torch 2.9.1 cu126 (supports T4 / sm75)
if not os.path.exists(f"{GEN}/.ready"):
    sh(f"uv venv -q --python 3.12 {GEN}")
    sh(f"uv pip install -q --python {GEN}/bin/python torch==2.9.1 torchaudio==2.9.1 --index-url https://download.pytorch.org/whl/cu126")
    sh(f"uv pip install -q --python {GEN}/bin/python transformers==5.23.0 diffusers==0.43.0 accelerate safetensors "
       f"huggingface_hub hf_xet soundfile librosa imageio imageio-ffmpeg sentencepiece pillow numpy "
       f"voicetut-tts==0.1.1 'omnivoice @ git+https://github.com/k2-fsa/OmniVoice.git@08be0b4ccbac3e13e374e86fbfead4b4cac343e2'")
    open(f"{GEN}/.ready", "w").write("ok")
done("2a_tts_install", t); t = time.time()
TTS_ENGINE = None
try:
    sh(f"{GEN}/bin/python {WORK}/stages/tts_stage.py"); TTS_ENGINE = "VoiceTut-TTS"
except Exception as e:
    print("⚠️ VoiceTut failed:", e, "\n→ trying Chatterbox Multilingual fallback")
    try:
        CB = "/content/venvs/chatterbox"
        if not os.path.exists(f"{CB}/bin/python"):
            sh(f"uv venv -q --python 3.11 {CB}")
            sh(f"uv pip install -q --python {CB}/bin/python chatterbox-tts==0.1.7")
        sh(f"{CB}/bin/python {WORK}/stages/tts_chatterbox.py"); TTS_ENGINE = "Chatterbox"
    except Exception as e2:
        print("⚠️ Chatterbox failed too:", e2, "\n→ using the original Gemini Aoede lines so the rest still runs")
        os.makedirs(f"{WORK}/tts", exist_ok=True)
        for i in range(1, len(script["lines"]) + 1):
            shutil.copy(f"{WORK}/inputs/aoede_line{i}.wav", f"{WORK}/tts/raw_line{i}.wav")
        TTS_ENGINE = "original-aoede"
sh(f"python3 {WORK}/stages/post_audio.py")
print("TTS engine:", TTS_ENGINE)
display(Audio(f"{OUT}/audio/vo_full.wav"))
done("2b_tts_generate", t)

In [ ]:
#@title 3) Stage 2 — image-to-video (LTX-Video 0.9.8 2B distilled) { display-mode: "form" }
from IPython.display import Image as IPImage
t = time.time()
sh(f"{GEN}/bin/python {WORK}/stages/i2v_stage.py")
sheet = f"{OUT}/clips/i2v_contact.jpg"
sh(f"ffmpeg -y -loglevel error " + " ".join(f"-i {OUT}/clips/i2v_scene{i}.mp4" for i in range(1, 5)) +
   " -filter_complex \"" + "".join(f"[{k}:v]select='not(mod(n\\,24))',scale=160:-2,tile=5x1[r{k}];" for k in range(4)) +
   "[r0][r1][r2][r3]vstack=4\" -frames:v 1 " + sheet, check=False)
if os.path.exists(sheet): display(IPImage(sheet, width=640))
try:
    for r in json.load(open(f"{OUT}/clips/i2v_report.json"))["scenes"]: print(r)
    err = open(f"{OUT}/clips/i2v_errors.log").read().strip()
    if err: print("--- i2v_errors.log (tail) ---\n" + err[-3000:])
except Exception as e:
    print("no i2v report:", e)
done("3_i2v", t)

In [ ]:
#@title 4) Stage 3 — lip-sync (LatentSync 1.5, fp16) { display-mode: "form" }
t = time.time()
LS, LSV = "/content/LatentSync", "/content/venvs/latentsync"
LS_OK = False
if CONFIG["lipsync"]:
    try:
        if not os.path.exists(f"{LS}/.ready"):
            if not os.path.exists(LS):
                sh(f"git clone -q https://github.com/bytedance/LatentSync {LS}")
            sh("git checkout -q a229c3948406bc2cf6eaf4873e662e70c6a04746", cwd=LS)
            sh(f"bash {WORK}/stages/ls_patch.sh {LS}")
            sh(f"uv venv -q --python 3.10 {LSV}")
            sh(f"uv pip install -q --python {LSV}/bin/python torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121")
            sh(f"grep -vE '^(gradio|torch|torchvision|--extra-index-url)' {LS}/requirements.txt > {LS}/req-colab.txt")
            sh(f"uv pip install -q --python {LSV}/bin/python -r {LS}/req-colab.txt")
            sh(f"{GEN}/bin/python -c \"from huggingface_hub import hf_hub_download as d; "
               f"[print(d('ByteDance/LatentSync-1.5', f, local_dir='{LS}/checkpoints')) for f in ('latentsync_unet.pt', 'whisper/tiny.pt')]\"")
            shutil.copy(f"{WORK}/stages/ls_runner.py", f"{LS}/ls_runner.py")
            open(f"{LS}/.ready", "w").write("ok")
        done("4a_lipsync_install", t); t = time.time()
        os.makedirs(f"{WORK}/ls_in", exist_ok=True); jobs = []
        for s in json.load(open(f"{WORK}/inputs/scenes.json"))["scenes"]:
            i = s["id"]; v = f"{WORK}/ls_in/scene{i}.mp4"
            # upscale 480x832 -> 720x1248 so the face crop has more pixels to work with
            sh(f"ffmpeg -y -loglevel error -i {OUT}/clips/i2v_scene{i}.mp4 -vf scale=720:1248:flags=lanczos,fps=25 -c:v libx264 -crf 14 -pix_fmt yuv420p {v}")
            jobs.append({"video": v, "audio": f"{OUT}/audio/line{s['line']}_16k.wav", "out": f"{OUT}/clips/lipsync_scene{i}.mp4"})
        json.dump(jobs, open(f"{WORK}/ls_jobs.json", "w"), indent=1)
        sh(f"{LSV}/bin/python ls_runner.py {WORK}/ls_jobs.json", cwd=LS,
           env={"LS_STEPS": str(CONFIG["ls_steps"]), "LS_GUIDANCE": str(CONFIG["ls_guidance"])})
        res = json.load(open(f"{WORK}/ls_jobs_result.json"))
        LS_OK = all(r["ok"] for r in res)
        for r in res: print(("✅" if r["ok"] else "⚠️ no lip-sync, will use plain clip:"), os.path.basename(r["out"]), r["seconds"], "s")
    except Exception as e:
        print("⚠️ Lip-sync stage failed → the reel will use the un-synced clips:", e)
done("4b_lipsync", t)

In [ ]:
#@title 5) Stage 4 — assemble, preview, download { display-mode: "form" }
from IPython.display import Video, HTML
t = time.time()
sh(f"python3 {WORK}/stages/assemble.py")
done("5_assemble", t)
FINAL = f"{OUT}/xmas-reel-colab.mp4"
report = {"times_sec": TIMES, "total_min": round((time.time() - T_START) / 60, 1), "tts_engine": TTS_ENGINE,
          "lipsync_all_ok": LS_OK, "config": CONFIG}
for f in ("clips/i2v_report.json",):
    try: report["i2v"] = json.load(open(f"{OUT}/{f}"))
    except Exception: pass
json.dump(report, open(f"{OUT}/report.json", "w"), indent=1, ensure_ascii=False)
print(json.dumps(report["times_sec"], indent=1)); print("TOTAL", report["total_min"], "min")
display(Video(FINAL, embed=True, width=360, html_attributes="controls loop"))
ZIP = shutil.make_archive("/content/xmas-reel-colab-out", "zip", OUT)   # reel + clips + audio + reports
print(f"ZIP: {ZIP} ({os.path.getsize(ZIP)/1e6:.1f} MB)")
from IPython.display import FileLink
try:
    from google.colab import files
    files.download(FINAL)
    time.sleep(3)              # two downloads back-to-back: give the browser a moment (allow "multiple downloads" if asked)
    files.download(ZIP)
except Exception as e:
    print("Automatic download failed:", e)
print("If a download didn't start: Files panel (left) → right-click → Download:\n ", FINAL, "\n ", ZIP)